# 3 — Risk differentiation : classes homogènes de risque

Ce notebook regroupe les scores d'un segment en **classes homogènes de risque (CHR)** : un petit nombre de classes ordonnées, chacune rassemblant des clients de risque comparable. Les CHR sont l'unité sur laquelle se fera la quantification du risque (chapitre 8) : chaque classe recevra une PD.

Le notebook est **paramétré par segment** : il suffit de changer `CODE` (`"SA"`, `"SB"` ou `"SM"`) pour l'appliquer à un autre segment, à condition que les notebooks 1 et 2 aient été exécutés pour ce segment. Il produit, pour chaque segment, une base au format commun (segment, score, PD du score, CHR) qui sera rassemblée avec celles des autres segments dans le notebook de quantification (échelle maître, chapitre 7.5 et chapitre 8).

**Une bonne mise en classes doit être :**
- **ordonnée** : le taux de défaut augmente d'une classe à la suivante ;
- **stable dans le temps** (7.2) : l'ordre des classes et leur volume se maintiennent d'un mois à l'autre ;
- **homogène à l'intérieur** (7.3) : le risque d'une classe ne varie pas selon d'autres caractéristiques des clients ;
- **hétérogène entre classes** (7.4) : deux classes voisines ont des risques clairement différents ;
- **fidèle au score** (7.6) : le regroupement perd peu de pouvoir discriminant par rapport au score continu.

In [1]:
# Cellule 1 — paramètres et chargement
from pathlib import Path
import polars as pl
from utils import binning, chr as chrm, exploration

pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_hide_column_data_types(True)

CODE = "SB"                       # segment étudié : "SA", "SB" ou "SM"
ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
VARIABLES_EXOGENES = ["ANCIENNETE_modif", "AGE_PP_EN_MOIS"]   # variables non retenues par la régression, pour l'homogénéité

base = pl.read_parquet(f"data/processed/base_score_{CODE}.parquet")
modele = pl.read_parquet(f"data/processed/base_modele_{CODE}.parquet")
VARIABLES_EXOGENES = [v for v in VARIABLES_EXOGENES if v in modele.columns]
base = base.join(modele.select(ID, DATE, *VARIABLES_EXOGENES), on=[ID, DATE], how="left")
regles_binning = binning.charger_regles(f"outputs/scorecard/regles_{CODE}.json")

train, test = base.filter(pl.col("echantillon") == "train"), base.filter(pl.col("echantillon") == "test")
print(f"{CODE} : {train.height} lignes train ({train[TARGET].sum():.0f} défauts) | {test.height} lignes test ({test[TARGET].sum():.0f} défauts)")
print(f"score de {base['score'].min():.0f} à {base['score'].max():.0f}, {base['score'].n_unique()} valeurs distinctes")

SB : 88629 lignes train (1091 défauts) | 37996 lignes test (470 défauts)
score de 444 à 720, 239 valeurs distinctes


## 3.1 Construction des classes (7.1)

Trois méthodes sont comparées, toutes estimées sur l'apprentissage :

| Méthode | Principe |
|---|---|
| **Quantiles + fusion** | découpage fin du score en 30 tranches de même effectif, puis fusion des tranches voisines qui ne respectent pas les règles de stabilité |
| **Jenks** (natural breaks) | coupures qui minimisent la variance de la PD à l'intérieur des classes, pour un nombre de classes k fixé ; la qualité se lit dans le GVF (*goodness of variance fit*, proche de 1 = classes homogènes) |
| **Jenks + fusion** | les coupures de Jenks (k = 10 à 15) servent de découpage fin, puis les classes voisines qui ne respectent pas les règles sont fusionnées |

Les **règles de fusion** sont celles du chapitre 4, appliquées cette fois au score : chaque classe doit compter au moins 5 % de la population et 20 défauts, le taux de défaut doit être monotone, et deux classes voisines doivent avoir un écart de risque d'au moins 30 % **sur chacun des mois**. Jenks seul ne garantit pas ces règles ; il est montré à titre de comparaison.

Le critère de choix est la **perte d'information** (7.6) : on compare, sur le test, le Gini du score continu et celui des CHR (chaque client recevant le taux de défaut de sa classe). On retient, parmi les méthodes qui respectent les règles, celle qui perd le moins de Gini.

**Découpage optimal (optbinning).** La librairie `optbinning` cherche, par programmation par contraintes, les bornes du score qui maximisent l'IV sous contraintes de monotonie, d'effectif (5 %) et de défauts minimaux par classe. Elle ne connaît pas la règle d'écart de 30 % chaque mois : on la teste seule puis suivie de la fusion. Une variante abaisse le minimum de défauts à 10 par classe, pour voir si la classe la moins risquée, qui rassemble plus de la moitié des clients, peut être scindée.

Une méthode respecte les règles si chaque classe compte au moins 5 % de la population et 20 défauts, et si deux classes voisines diffèrent d'au moins 30 % en risque **chaque mois**.

In [2]:
# Cellule 2 — comparaison des méthodes de construction
PARAMS_CHR = dict(part_min=0.05, defauts_min=20, ecart_min=0.3)
candidats, lignes = {}, []

candidats["quantiles + fusion"] = chrm.chr_par_regroupement(train, "score", TARGET, DATE, n_fin=30, **PARAMS_CHR)
for k in range(5, 13):
    regle_k, gvf = chrm.chr_jenks(train, "score", "pd_score", k)
    candidats[f"jenks k={k}"] = regle_k | {"gvf": gvf}
for k in (10, 12, 15):
    regle_k, _ = chrm.chr_jenks(train, "score", "pd_score", k)
    candidats[f"jenks k={k} + fusion"] = chrm.chr_par_regroupement(train, "score", TARGET, DATE, bornes_fines=regle_k["bornes"], **PARAMS_CHR)
for kmax in (None, 6, 8):
    for dmin in (20, 10):
        regle_o = chrm.chr_optbinning(train, "score", TARGET, PARAMS_CHR["part_min"], dmin, max_n_bins=kmax)
        candidats[regle_o["methode"]] = regle_o
        candidats[regle_o["methode"] + " + fusion"] = chrm.chr_par_regroupement(
            train, "score", TARGET, DATE, bornes_fines=regle_o["bornes"], **PARAMS_CHR)

for nom, regle in candidats.items():
    perte = chrm.perte_information(test, "score", TARGET, regle)
    stab, psi = chrm.stabilite_chr(train, "score", TARGET, DATE, regle)
    ecarts = stab["ecart_min_suivante"].drop_nulls().drop_nans()
    t_chr = chrm.table_chr(train, "score", TARGET, regle)
    lignes.append({"methode": nom, "n_classes": len(regle["bornes"]) + 1,
                   "regles_respectees": chrm.regles_chr_respectees(train, "score", TARGET, DATE, regle, **PARAMS_CHR),
                   "gvf": regle.get("gvf"), "part_max": t_chr["part_population"].max(), "defauts_min_classe": t_chr["n_defaut"].min(),
                   "ecart_min_mois": ecarts.min() if ecarts.len() else None, "psi_max": psi,
                   "gini_score_test": perte["gini_score"], "gini_chr_test": perte["gini_chr"], "perte_points": perte["perte_points"]})
comparaison_chr = pl.DataFrame(lignes).unique("methode", keep="first", maintain_order=True)
print(comparaison_chr)

shape: (24, 11)
┌─────────────────────────────────┬───────────┬───────────────────┬──────────┬──────────┬────────────────────┬────────────────┬──────────┬─────────────────┬───────────────┬──────────────┐
│ methode                         ┆ n_classes ┆ regles_respectees ┆ gvf      ┆ part_max ┆ defauts_min_classe ┆ ecart_min_mois ┆ psi_max  ┆ gini_score_test ┆ gini_chr_test ┆ perte_points │
╞═════════════════════════════════╪═══════════╪═══════════════════╪══════════╪══════════╪════════════════════╪════════════════╪══════════╪═════════════════╪═══════════════╪══════════════╡
│ quantiles + fusion              ┆ 4         ┆ true              ┆ null     ┆ 0.659367 ┆ 55.0               ┆ 0.947397       ┆ 0.001045 ┆ 0.824759        ┆ 0.77687       ┆ 4.788895     │
│ jenks k=5                       ┆ 5         ┆ false             ┆ 0.939455 ┆ 0.898318 ┆ 97.0               ┆ 0.7251         ┆ 0.00021  ┆ 0.824759        ┆ 0.652388      ┆ 17.237099    │
│ jenks k=6                       ┆ 6       

In [3]:
# Cellule 3 — CHR retenues : méthode respectant les règles avec la plus faible perte de Gini
CHOIX = comparaison_chr.filter(pl.col("regles_respectees")).sort("perte_points")["methode"][0]
regle_chr = {k: v for k, v in candidats[CHOIX].items() if k != "gvf"}
print("méthode retenue :", CHOIX, "| bornes de score :", regle_chr["bornes"])

table_train = chrm.table_chr(train, "score", TARGET, regle_chr, "pd_score")
table_test = chrm.table_chr(test, "score", TARGET, regle_chr, "pd_score")
print("Apprentissage"); print(table_train)
print("Test"); print(table_test)

méthode retenue : optbinning défauts≥10 + fusion | bornes de score : [571.5, 607.5, 634.5, 653.5]
Apprentissage
shape: (5, 9)
┌─────┬──────────────────┬───────┬─────────────────┬──────────┬─────────────┬──────────┬──────────┬────────────────────┐
│ chr ┆ intervalle_score ┆ n     ┆ part_population ┆ n_defaut ┆ taux_defaut ┆ ic_bas   ┆ ic_haut  ┆ pd_predite_moyenne │
╞═════╪══════════════════╪═══════╪═════════════════╪══════════╪═════════════╪══════════╪══════════╪════════════════════╡
│ 1   ┆ ]653.5 ; +inf]   ┆ 57135 ┆ 0.644654        ┆ 51.0     ┆ 0.000893    ┆ 0.000545 ┆ 0.001307 ┆ 0.001051           │
│ 2   ┆ ]634.5 ; 653.5]  ┆ 10142 ┆ 0.114432        ┆ 48.0     ┆ 0.004733    ┆ 0.003572 ┆ 0.006269 ┆ 0.004504           │
│ 3   ┆ ]607.5 ; 634.5]  ┆ 9038  ┆ 0.101976        ┆ 94.0     ┆ 0.010401    ┆ 0.008507 ┆ 0.01271  ┆ 0.009805           │
│ 4   ┆ ]571.5 ; 607.5]  ┆ 7771  ┆ 0.08768         ┆ 251.0    ┆ 0.0323      ┆ 0.028594 ┆ 0.036467 ┆ 0.028987           │
│ 5   ┆ ]-inf ; 571.5]   ┆ 

**Résultats.** La méthode retenue est le découpage en quantiles suivi de la fusion des classes voisines : elle produit **5 CHR**, avec un Gini de 0,790 sur le test contre 0,815 pour le score continu, soit une perte limitée à 2,5 points. Jenks seul perd un peu moins d'information avec 11 ou 12 classes, mais ses classes se croisent certains mois (écart minimal négatif) et ne sont donc pas exploitables ; combiné à la fusion, il aboutit à 3 ou 4 classes et perd davantage.

Le taux de défaut croît fortement d'une classe à l'autre, de 0,05 % (classe 1, 53 % de la population) à 11 % (classe 5, 7 % de la population), et les taux observés sur le test sont proches de ceux de l'apprentissage. La classe 1 rassemble plus de la moitié du segment : elle regroupe notamment les clients au score maximal, que le score ne permet pas de distinguer entre eux.


## 3.2 Stabilité dans le temps (7.2)

Par construction, les classes sont ordonnées sur l'ensemble de l'apprentissage. Mais **mois par mois**, deux classes voisines peuvent se croiser (la classe 3 moins risquée que la classe 2 en avril, par exemple). Le tableau donne le taux de défaut de chaque classe pour chaque mois et, dans `ecart_min_suivante`, le plus petit écart relatif avec la classe voisine sur l'ensemble des mois : une valeur négative signale un croisement. La stabilité en **volume** est mesurée par le PSI de la répartition des classes d'un mois à l'autre (< 0,1 : stable). Le contrôle est fait sur l'apprentissage et sur le test.

In [4]:
# Cellule 4 — stabilité en risque (croisements par mois) et en volume (PSI)
for nom, d in (("Apprentissage", train), ("Test", test)):
    stab, psi = chrm.stabilite_chr(d, "score", TARGET, DATE, regle_chr)
    print(f"{nom} — PSI maximal entre mois : {psi:.4f}"); print(stab)

Apprentissage — PSI maximal entre mois : 0.0012
shape: (5, 6)
┌─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬────────────────────┐
│ classe          ┆ taux_2021-01-01 ┆ taux_2021-02-01 ┆ taux_2021-03-01 ┆ taux_2021-04-01 ┆ ecart_min_suivante │
╞═════════════════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╪════════════════════╡
│ ]-inf ; 571.5]  ┆ 0.131381        ┆ 0.148148        ┆ 0.138233        ┆ 0.153005        ┆ 3.086626           │
│ ]571.5 ; 607.5] ┆ 0.030992        ┆ 0.030678        ┆ 0.033826        ┆ 0.033546        ┆ 1.336652           │
│ ]607.5 ; 634.5] ┆ 0.010673        ┆ 0.013129        ┆ 0.010029        ┆ 0.00771         ┆ 0.884067           │
│ ]634.5 ; 653.5] ┆ 0.00362         ┆ 0.006235        ┆ 0.005323        ┆ 0.003659        ┆ 2.484146           │
│ ]653.5 ; +inf]  ┆ 0.000972        ┆ 0.000695        ┆ 0.000854        ┆ 0.00105         ┆ NaN                │
└─────────────────┴───────────────

**Résultats.** Aucun croisement de classes n'apparaît, ni sur l'apprentissage ni sur le test : sur chacun des quatre mois, chaque classe reste au moins 63 % plus risquée que sa voisine. Le PSI maximal entre mois (0,0008) montre que la répartition des clients entre classes est parfaitement stable.


## 3.3 Hétérogénéité entre classes voisines (7.4)

On vérifie que deux classes voisines ont des risques nettement différents. Avec de grands effectifs, une p-value seule est peu informative (le moindre écart devient significatif) : on rapporte aussi des **mesures d'ampleur**.
- **Rapport des taux de défaut** : la classe la plus risquée doit l'être au moins 1,3 fois plus (règle des 30 %).
- **Log odds ratio** : écart de risque sur l'échelle de la régression logistique, adapté aux événements rares.
- **h de Cohen** : taille d'effet usuelle pour comparer deux proportions. **Limite** : pour des taux de défaut de l'ordre du pour cent, h reste faible (< 0,2) même quand le risque double, car il est construit sur l'échelle arcsin √p où les petites proportions sont tassées. Les seuils de Cohen (0,2 / 0,5 / 0,8) ne s'appliquent donc pas aux événements rares ; h est donné pour information, le rapport et le log odds ratio étant les mesures pertinentes ici.
- **Recouvrement des intervalles de Wilson** : si les intervalles de confiance de deux classes voisines se chevauchent, elles ne sont pas clairement séparées.
- **Test de différence de deux proportions** (unilatéral), en complément.

Le contrôle est fait sur le test, échantillon qui n'a pas servi à construire les classes.

In [5]:
# Cellule 5 — hétérogénéité entre classes voisines (test)
print(chrm.heterogeneite_chr(table_test))

shape: (4, 8)
┌───────┬──────────┬──────────┬──────────┬──────────┬────────────────┬──────────────────┬────────────┐
│ paire ┆ taux_1   ┆ taux_2   ┆ rapport  ┆ h_cohen  ┆ log_odds_ratio ┆ ic_se_recouvrent ┆ p_value    │
╞═══════╪══════════╪══════════╪══════════╪══════════╪════════════════╪══════════════════╪════════════╡
│ 1 → 2 ┆ 0.000979 ┆ 0.004915 ┆ 5.019424 ┆ 0.077731 ┆ 1.617262       ┆ false            ┆ 9.3468e-10 │
│ 2 → 3 ┆ 0.004915 ┆ 0.010781 ┆ 2.193669 ┆ 0.067715 ┆ 0.791488       ┆ false            ┆ 0.001285   │
│ 3 → 4 ┆ 0.010781 ┆ 0.032735 ┆ 3.036338 ┆ 0.15582  ┆ 1.133095       ┆ false            ┆ 4.6182e-11 │
│ 4 → 5 ┆ 0.032735 ┆ 0.138548 ┆ 4.232459 ┆ 0.398942 ┆ 1.558637       ┆ false            ┆ 2.6147e-48 │
└───────┴──────────┴──────────┴──────────┴──────────┴────────────────┴──────────────────┴────────────┘


**Résultats.** Les classes voisines sont nettement séparées sur le test : le taux de défaut est multiplié par 2,2 à 5 d'une classe à la suivante (log odds ratio de 0,8 à 1,6), les intervalles de confiance de Wilson ne se recouvrent jamais et toutes les différences sont significatives. Le h de Cohen reste compris entre 0,07 et 0,27 malgré ces écarts importants, ce qui illustre sa limite pour des taux de défaut faibles.


## 3.4 Homogénéité à l'intérieur des classes (7.3)

Une classe est homogène si son taux de défaut ne dépend pas d'autres caractéristiques des clients : sinon, la classe mélange des populations de risques différents et une PD unique par classe serait biaisée. On le vérifie avec des variables **extérieures au score** : le mois d'observation et des variables non retenues par la régression (`ANCIENNETE_modif`, `AGE_PP_EN_MOIS`), découpées selon leurs classes du notebook 1. Pour chaque CHR, on mesure l'écart maximal de taux de défaut entre sous-groupes (h de Cohen, avec la même limite que ci-dessus) et on teste l'indépendance (Chi²). Les sous-groupes de moins de 30 dossiers sont ignorés.

In [6]:
# Cellule 6 — homogénéité interne selon le mois et des variables extérieures au score (apprentissage)
homogeneite = [chrm.homogeneite_chr(train, "score", TARGET, regle_chr, DATE)]
for v in VARIABLES_EXOGENES:
    homogeneite.append(chrm.homogeneite_chr(train, "score", TARGET, regle_chr, v, regles_binning.get(v)))
homogeneite = pl.concat([h for h in homogeneite if h.height], how="diagonal_relaxed")
print(homogeneite)

shape: (15, 7)
┌─────┬──────────────────┬────────────────┬──────────┬──────────┬─────────────┬──────────┐
│ chr ┆ variable         ┆ n_sous_groupes ┆ taux_min ┆ taux_max ┆ h_cohen_max ┆ p_chi2   │
╞═════╪══════════════════╪════════════════╪══════════╪══════════╪═════════════╪══════════╡
│ 1   ┆ datdelhis        ┆ 4              ┆ 0.000695 ┆ 0.00105  ┆ 0.012106    ┆ 0.763392 │
│ 2   ┆ datdelhis        ┆ 4              ┆ 0.00362  ┆ 0.006235 ┆ 0.037683    ┆ 0.443557 │
│ 3   ┆ datdelhis        ┆ 4              ┆ 0.00771  ┆ 0.013129 ┆ 0.053832    ┆ 0.354436 │
│ 4   ┆ datdelhis        ┆ 4              ┆ 0.030678 ┆ 0.033826 ┆ 0.017821    ┆ 0.916245 │
│ 5   ┆ datdelhis        ┆ 4              ┆ 0.131381 ┆ 0.153005 ┆ 0.061959    ┆ 0.44686  │
│ 1   ┆ ANCIENNETE_modif ┆ 4              ┆ 0.00074  ┆ 0.001254 ┆ 0.016434    ┆ 0.410283 │
│ 2   ┆ ANCIENNETE_modif ┆ 4              ┆ 0.002527 ┆ 0.005581 ┆ 0.048962    ┆ 0.541782 │
│ 3   ┆ ANCIENNETE_modif ┆ 4              ┆ 0.007681 ┆ 0.017585 ┆ 0.090493 

**Résultats.** Les classes sont homogènes selon le mois d'observation (p > 0,3 pour toutes les classes). En revanche, dans la classe la plus risquée (classe 5), le taux de défaut varie selon l'ancienneté de la relation (de 6,4 % à 12,1 %, p = 0,002) et selon l'âge (de 6,1 % à 12,3 %, p = 0,0003) ; un effet plus faible apparaît dans la classe 3. Ces deux variables ont été écartées par la sélection de la régression, leur effet n'étant pas significatif une fois les autres variables présentes : leur influence est concentrée sur les clients les plus risqués (interaction que la régression additive ne capte pas). Cette hétérogénéité modérée est documentée comme une limite. Elle ne conduit pas à découper la classe 5 selon ces variables, les CHR devant rester définies par le score, et l'âge est en outre un critère sensible au regard de la non-discrimination.


## 3.4 bis Défauts mal classés (« faux bons »)

Certains clients sont notés dans une CHR peu risquée (1 ou 2) et font pourtant défaut. On les étudie **de façon agrégée** :
- **combien** : part des défauts dans chaque CHR ;
- **ce qui les distingue** : au sein des CHR 1 et 2, chaque variable découpée au notebook 1 (retenue ou non par le modèle) est comparée entre défauts et sains, par son IV calculée dans ces seules classes. Une variable **hors modèle** avec une IV élevée signale un risk driver que le score ne capte pas pour ces clients ; elle pourrait aussi servir à scinder la CHR 1, qui rassemble plus de la moitié de la population.

Avec environ 175 défauts dans ces classes (apprentissage et test réunis), l'analyse reste descriptive : les IV calculées sur si peu de défauts sont fragiles.

In [7]:
# Cellule 6 bis — faux bons : défauts classés dans les CHR 1 et 2 (apprentissage + test)
numeros = dict(zip(table_train["intervalle_score"], table_train["chr"]))
complet = pl.read_parquet(f"data/processed/base_modele_{CODE}.parquet")
d = (base.select(ID, DATE, TARGET, "score")
     .with_columns(chr=binning.appliquer_regle(base, "score", regle_chr)["classe"].replace_strict(numeros, return_dtype=pl.Int64))
     .join(complet.drop(TARGET, strict=False), on=[ID, DATE], how="left"))
print(d.group_by("chr").agg(n=pl.len(), n_defaut=pl.col(TARGET).sum())
      .with_columns(part_defauts=pl.col("n_defaut") / pl.col("n_defaut").sum()).sort("chr"))

VARS_SCORE = list(binning.charger_regles(f"outputs/scorecard/regles_modele_{CODE}.json"))
faux_bons = chrm.profil_faux_bons(d, TARGET, "chr", [1, 2], regles_binning, VARS_SCORE)
print(faux_bons.head(15))

shape: (5, 4)
┌─────┬───────┬──────────┬──────────────┐
│ chr ┆ n     ┆ n_defaut ┆ part_defauts │
╞═════╪═══════╪══════════╪══════════════╡
│ 1   ┆ 81647 ┆ 75.0     ┆ 0.048046     │
│ 2   ┆ 14415 ┆ 69.0     ┆ 0.044202     │
│ 3   ┆ 12841 ┆ 135.0    ┆ 0.086483     │
│ 4   ┆ 11223 ┆ 364.0    ┆ 0.233184     │
│ 5   ┆ 6499  ┆ 918.0    ┆ 0.588085     │
└─────┴───────┴──────────┴──────────────┘
shape: (15, 7)
┌───────────────────────────────┬─────────────┬────────────────────┬────────────────────────┬────────────────────┬────────────────────────┬─────────────────┐
│ variable                      ┆ dans_modele ┆ iv_dans_bonnes_chr ┆ classe_la_plus_risquee ┆ taux_defaut_classe ┆ part_population_classe ┆ n_defaut_classe │
╞═══════════════════════════════╪═════════════╪════════════════════╪════════════════════════╪════════════════════╪════════════════════════╪═════════════════╡
│ CRTAD_AG_NBJDE_BB             ┆ false       ┆ 0.545678           ┆ ]77 ; +inf]            ┆ 0.007198           ┆ 0.02

## 3.5 Export

On sauvegarde la règle de découpage du score en CHR et une base au **format commun à tous les segments** (identifiant, date, cible, échantillon, segment, score, PD du score, numéro de CHR). Le notebook de quantification rassemblera les bases des segments disponibles pour construire l'échelle maître et calibrer les PD.

In [8]:
# Cellule 7 — export : règle des CHR, base au format commun, tableaux de contrôle
binning.sauver_regles({"chr": regle_chr}, f"outputs/chr/regle_chr_{CODE}.json")

numeros = dict(zip(table_train["intervalle_score"], table_train["chr"]))
base_chr = base.select(ID, DATE, TARGET, "echantillon", "segment", "score", "pd_score").with_columns(
    chr=binning.appliquer_regle(base, "score", regle_chr)["classe"].replace_strict(numeros, return_dtype=pl.Int64))
base_chr.write_parquet(f"data/processed/base_chr_{CODE}.parquet")

exploration.exporter_excel({
    f"{CODE}_Comparaison": comparaison_chr,
    f"{CODE}_CHR_train": table_train, f"{CODE}_CHR_test": table_test,
    f"{CODE}_Heterogeneite": chrm.heterogeneite_chr(table_test),
    f"{CODE}_Homogeneite": homogeneite,
}, f"outputs/chr/chr_{CODE}.xlsx")
print(base_chr.group_by("echantillon", "chr").agg(pl.len(), pl.col(TARGET).mean()).sort("echantillon", "chr"))

shape: (10, 4)
┌─────────────┬─────┬───────┬─────────────┐
│ echantillon ┆ chr ┆ len   ┆ ddefaut_ndb │
╞═════════════╪═════╪═══════╪═════════════╡
│ test        ┆ 1   ┆ 24512 ┆ 0.000979    │
│ test        ┆ 2   ┆ 4273  ┆ 0.004915    │
│ test        ┆ 3   ┆ 3803  ┆ 0.010781    │
│ test        ┆ 4   ┆ 3452  ┆ 0.032735    │
│ test        ┆ 5   ┆ 1956  ┆ 0.138548    │
│ train       ┆ 1   ┆ 57135 ┆ 0.000893    │
│ train       ┆ 2   ┆ 10142 ┆ 0.004733    │
│ train       ┆ 3   ┆ 9038  ┆ 0.010401    │
│ train       ┆ 4   ┆ 7771  ┆ 0.0323      │
│ train       ┆ 5   ┆ 4543  ┆ 0.142417    │
└─────────────┴─────┴───────┴─────────────┘
